# Tập 3.1 · "Tôi đang ở đâu?": bộ lọc Bayes 🚪

[![Mở trên Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thanhpro82/tu-hoc-robot-tu-hanh/blob/main/tap31/tap31_bayes.ipynb)

Series **Tự Học Robot Tự Hành** · Mùa 3 · ⏱ ~35 phút · Giáo trình: Principles of Robot Autonomy, [Chương 11](https://porabook.com/book/localization-filtering/)

Robot trong hành lang có 3 cửa, giữ một niềm tin xác suất về vị trí của mình. Trong notebook này bạn:

1. Chạy lại đúng mô phỏng trong video.
2. **Tự viết** bước hiệu chỉnh, bước dự đoán và cả bộ lọc Bayes (3 bài), có ô chấm tự động.
3. Vẽ niềm tin ở 4 thời điểm bằng bộ lọc của bạn.

Chạy lần lượt từng ô từ trên xuống (Shift + Enter).

## 0. Chuẩn bị (chạy 1 lần)

In [ ]:
import os, sys
if "google.colab" in sys.modules:  # trên Colab: tải code về
    if not os.path.exists("/content/tu-hoc-robot-tu-hanh"):
        !git clone -q https://github.com/thanhpro82/tu-hoc-robot-tu-hanh.git /content/tu-hoc-robot-tu-hanh
    os.chdir("/content/tu-hoc-robot-tu-hanh/tap31")
sys.path[:0] = ["bai_tap", "sim", ".."]

import numpy as np
import bayes as goc                # mô phỏng trong video
from ve_do_thi import ve           # vẽ hình
from IPython.display import Image, display
from kiem_tra import cham          # bộ chấm
print("Sẵn sàng!")

## 1. Chạy lại số liệu trong video

In [ ]:
for r in goc.LOG:
    if r["loai"] != "hiệu chỉnh":
        continue
    b = r["bel"]
    z = "thấy cửa" if r["z"] else "không thấy"
    print(f"bước {r['buoc']:2d} x thật {r['x']:.2f} m | {z:10s} | số đỉnh {goc.so_dinh(b)} | ±25 cm quanh vị trí thật: {goc.p_quanh(b, r['x']) * 100:5.1f}%")

## 2. Bài tập

Dự đoán: $\bar p_k = \sum_i p(x_k \mid x_i, u)\,p_i$ với $p(x_k \mid x_i, u) = \mathcal N(x_k - x_i;\ 0{,}5,\ 0{,}1^2)$. Hiệu chỉnh: $p_k = \eta\, p(z \mid x_k)\,\bar p_k$.

| Bài | Hàm | Việc |
|---|---|---|
| **1** | `hieu_chinh(bel_tr, thay_cua)` | nhân với $p(z \mid x)$, chuẩn hoá; trả về `(bel, η)` |
| **2** | `du_doan(bel)` | `K @ bel` với ma trận chuẩn K, rồi chuẩn hoá |
| **3** | `loc_bayes(ds_z)` | bắt đầu đều; mỗi bước dự đoán (k > 0) rồi hiệu chỉnh |

Thay từng dòng `raise NotImplementedError(...)` bằng code của bạn, chạy ô đó, rồi chạy ô chấm ngay bên dưới (✅ đạt · ❌ sai, kèm gợi ý · ⏳ chưa làm).

In [ ]:
import numpy as np

DAI, O = 10.0, 0.05
X = np.arange(0, DAI, O) + O / 2
CUA = [(1.0, 1.8), (2.6, 3.4), (6.2, 7.0)]
O_CUA = np.zeros_like(X, dtype=bool)
for _a, _b in CUA:
    O_CUA |= (X >= _a) & (X <= _b)
P_THAY_O_CUA, P_THAY_KHONG_CUA = 0.8, 0.1
U, SIGMA = 0.5, 0.1        # mỗi bước đi u = 0,5 m, nhiễu chuyển động σ = 0,1 m


def hieu_chinh(bel_tr, thay_cua):
    """Bài 1 (bước hiệu chỉnh): p_k = η · p(z | x_k) · p̄_k.

    p(z | x_k): nếu ô k ở cửa thì p(thấy) = 0,8, không ở cửa thì p(thấy) = 0,1.
    Nếu z là "không thấy" (thay_cua = False) thì dùng 1 − p(thấy).
    η = 1 / Σ_k p(z | x_k)·p̄_k để tổng lại bằng 1. Trả về (bel_moi, eta)."""
    raise NotImplementedError("TODO Bài 1: hiệu chỉnh")


def du_doan(bel):
    """Bài 2 (bước dự đoán): p̄_k = Σ_i p(x_k | x_i, u)·p_i, với p(x_k | x_i, u) là mật độ chuẩn N(x_k − x_i; U, SIGMA²)·O.

    Gợi ý: d = X[:, None] − X[None, :] (ma trận x_k − x_i), K = exp(−½·((d − U)/SIGMA)²) / (√(2π)·SIGMA/O),
    p̄ = K @ bel. Phần xác suất "đi ra ngoài hành lang" bị mất, nên chia cho tổng để chuẩn hoá lại. Trả về p̄."""
    raise NotImplementedError("TODO Bài 2: dự đoán")


def loc_bayes(ds_z):
    """Bài 3: chạy cả bộ lọc. ds_z[k] là kết quả cảm biến (True/False) ở bước k = 0, 1, ..., n.

    Bắt đầu: bel đều = 1/200 cho mọi ô (chưa biết gì). Mỗi bước k: nếu k > 0 thì du_doan trước, rồi hieu_chinh với ds_z[k].
    Trả về list niềm tin SAU hiệu chỉnh của từng bước (n + 1 mảng)."""
    raise NotImplementedError("TODO Bài 3: vòng lặp bộ lọc Bayes")


In [ ]:
cham(globals())

## 3. Xem kết quả

In [ ]:
import loi_giai
try:
    ve(globals(), "ket_qua.png")
except NotImplementedError:
    print("Chưa xong cả 3 bài, vẽ bằng lời giải.")
    ve(loi_giai, "ket_qua.png")
display(Image("ket_qua.png"))

## 4. Thử thách

Chỉ đi mà không nhìn (bỏ hiệu chỉnh sau bước 0): niềm tin nhoè đi thế nào?

In [ ]:
import loi_giai
from kiem_tra import DS_Z
bel, _ = loi_giai.hieu_chinh(np.full_like(goc.X, 1 / len(goc.X)), True)
for k in range(1, 13):
    bel = loi_giai.du_doan(bel)
    if k in (1, 4, 8, 12):
        print(f"sau {k:2d} bước chỉ đi: đỉnh cao nhất {bel.max() * 100:.2f}%, số đỉnh {goc.so_dinh(bel)}")

## Câu hỏi tự kiểm tra

Tự trả lời trước, rồi mở đáp án ở mục *Câu hỏi tự kiểm tra* trong [README của tập](https://github.com/thanhpro82/tu-hoc-robot-tu-hanh/tree/main/tap31).

Code cả series: [github.com/thanhpro82/tu-hoc-robot-tu-hanh](https://github.com/thanhpro82/tu-hoc-robot-tu-hanh)